<a href="https://colab.research.google.com/github/MrJames509/Tarea-procesamiento/blob/main/Optimizaci%C3%B3n%20del%20Cronograma%20de%20una%20Vivienda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pygad numpy matplotlib

In [ ]:
import pygad
import numpy as np
import matplotlib.pyplot as plt

# 1. MODELADO DE DATOS (De la Imagen 3)
TAREAS_OBRA = {
    0: (5, 4),  # Cimientos
    1: (7, 6),  # Muros
    2: (4, 5),  # Techo
    3: (6, 3),  # Instalaciones
    4: (5, 4),  # Revoque
    5: (4, 3),  # Pisos
    6: (3, 2),  # Pintura
    7: (2, 3),  # Aberturas
    8: (4, 2),  # Electricidad
    9: (2, 2)   # Final de obra
}

MAX_OPERARIOS = 10
NUM_TAREAS = len(TAREAS_OBRA)

# 2. DISEÑO DE LA FUNCIÓN DE APTITUD (FITNESS)
def fitness_func(ga_instance, solution, solution_idx):
    """
    solution: Es un vector de 10 enteros, donde solution[i] es el día de inicio de la tarea i.
    Ejemplo: [0, 5, 2, ...] -> Tarea 0 inicia día 0, Tarea 1 inicia día 5, etc.
    """

    # Determinamos un horizonte de tiempo máximo razonable (suma de todas las duraciones)
    max_dias_posibles = sum(duracion for duracion, _ in TAREAS_OBRA.values())

    # Array para rastrear cuántos operarios trabajan cada día
    # Inicializamos con ceros
    operarios_por_dia = np.zeros(max_dias_posibles + 10)

    makespan = 0 # Día final de la obra
    penalizacion = 0

    # Recorremos cada tarea según el cromosoma (solución)
    for i in range(NUM_TAREAS):
        dia_inicio = int(solution[i])
        duracion, operarios_necesarios = TAREAS_OBRA[i]
        dia_fin = dia_inicio + duracion

        if dia_fin > makespan:
            makespan = dia_fin

        # Actualizamos el uso de operarios en el cronograma
        for d in range(dia_inicio, dia_fin):
            if d < len(operarios_por_dia):
                operarios_por_dia[d] += operarios_necesarios

    # VERIFICACIÓN DE RESTRICCIONES (La parte importante del desafío)
    max_operarios_usados = np.max(operarios_por_dia)

    if max_operarios_usados > MAX_OPERARIOS:
        # PENALIZACIÓN DRÁSTICA
        # Si se rompe la regla de los 10 operarios, el fitness debe ser muy bajo (cercano a 0)
        # para que el algoritmo descarte esta solución.
        fitness = 1 / (makespan * 1000)
    else:
        # Si es válido, optimizamos para minimizar el tiempo (maximizar 1/tiempo)
        fitness = 1 / makespan

    return fitness

# 3. CONFIGURACIÓN DEL ALGORITMO GENÉTICO (GA)
ga_instance = pygad.GA(
    num_generations=100,          # Cantidad de generaciones (evoluciones)
    num_parents_mating=10,        # Cuántos padres seleccionamos para cruzar
    sol_per_pop=50,               # Tamaño de la población (50 cronogramas distintos)
    num_genes=NUM_TAREAS,         # 10 genes (uno por tarea)

    # Usamos enteros. El espacio de genes es de 0 a 40 días (rango razonable)
    gene_type=int,
    gene_space=[range(0, 40) for _ in range(NUM_TAREAS)],

    fitness_func=fitness_func,

    # Parámetros de evolución
    parent_selection_type="sss",  # Steady State Selection
    keep_parents=1,               # Elitismo: guardamos al mejor de la generación anterior
    crossover_type="single_point",
    mutation_type="random",
    mutation_percent_genes=10,    # Probabilidad de mutación del 10%

    save_best_solutions=True,
    random_seed=42                # Para reproducibilidad
)

# 4. EJECUCIÓN Y MONITOREO
print("Iniciando optimización del cronograma...")
ga_instance.run()

# 5. ANÁLISIS DE RESULTADOS
solution, solution_fitness, solution_idx = ga_instance.best_solution(ga_instance.last_generation_fitness)

print("\n" + "="*40)
print("RESULTADOS ÓPTIMOS ENCONTRADOS")
print("="*40)

# Calculamos el Makespan real y verificamos operarios para el reporte
max_dias = sum(d for d, _ in TAREAS_OBRA.values())
operarios_dia = np.zeros(max_dias + 10)
makespan_final = 0

print(f"\n{'Tarea':<15} {'Inicio':<10} {'Fin':<10} {'Operarios':<10}")
print("-" * 50)

# Ordenamos las tareas por día de inicio para mostrar el "orden secuencial"
tareas_ordenadas = sorted([(i, int(solution[i])) for i in range(NUM_TAREAS)], key=lambda x: x[1])

for idx_tarea, dia_inicio in tareas_ordenadas:
    duracion, ops = TAREAS_OBRA[idx_tarea]
    dia_fin = dia_inicio + duracion
    if dia_fin > makespan_final:
        makespan_final = dia_fin

    # Actualizar conteo para verificación
    for d in range(dia_inicio, dia_fin):
        operarios_dia[d] += ops

    nombre_tarea = list(TAREAS_OBRA.keys())[idx_tarea] # Usamos el ID como nombre simple
    print(f"Tarea {idx_tarea:<13} {dia_inicio:<10} {dia_fin:<10} {ops:<10}")

print("-" * 50)
print(f"TIEMPO TOTAL DE OBRA (Makespan): {makespan_final} días")
print(f"Fitness obtenido: {solution_fitness:.4f}")
print(f"Máximo de operarios usados en un día: {int(np.max(operarios_dia))} (Límite: 10)")

# Gráfica de convergencia
ga_instance.plot_fitness()
plt.title("Evolución del Fitness (1 / Días Totales)")
plt.xlabel("Generación")
plt.ylabel("Fitness")
plt.show()

In [ ]:
# Datos del problema
TAREAS_OBRA = {
    0: (5, 4),   # Cimientos: 5 días, 4 operarios
    1: (7, 6),   # Muros: 7 días, 6 operarios
    2: (4, 5),   # Techo: 4 días, 5 operarios
    3: (6, 3),   # Instalaciones: 6 días, 3 operarios
    4: (5, 4),   # Revoque: 5 días, 4 operarios
    5: (4, 3),   # Pisos: 4 días, 3 operarios
    6: (3, 2),   # Pintura: 3 días, 2 operarios
    7: (2, 3),   # Aberturas: 2 días, 3 operarios
    8: (4, 2),   # Electricidad: 4 días, 2 operarios
    9: (2, 2)    # Final de obra: 2 días, 2 operarios
}

MAX_OPERARIOS = 10
NUM_TAREAS = len(TAREAS_OBRA)

# Función de fitness
def fitness_func(ga_instance, solution, solution_idx):
    max_dias = sum(duracion for duracion, _ in TAREAS_OBRA.values())
    operarios_por_dia = np.zeros(max_dias + 10)
    makespan = 0

    for i in range(NUM_TAREAS):
        dia_inicio = int(solution[i])
        duracion, operarios_nec = TAREAS_OBRA[i]
        dia_fin = dia_inicio + duracion

        if dia_fin > makespan:
            makespan = dia_fin

        for d in range(dia_inicio, dia_fin):
            if d < len(operarios_por_dia):
                operarios_por_dia[d] += operarios_nec

    # Penalización si excede operarios
    if np.max(operarios_por_dia) > MAX_OPERARIOS:
        return 1 / (makespan * 1000)
    else:
        return 1 / makespan

# Configuración del GA
ga_instance = pygad.GA(
    num_generations=100,
    num_parents_mating=10,
    sol_per_pop=50,
    num_genes=NUM_TAREAS,
    gene_type=int,
    gene_space=[range(0, 30) for _ in range(NUM_TAREAS)],
    fitness_func=fitness_func,
    parent_selection_type="sss",
    keep_parents=1,
    crossover_type="single_point",
    mutation_type="random",
    mutation_percent_genes=10,
    random_seed=42
)

# Ejecutar
ga_instance.run()

# Mostrar resultados
solution, solution_fitness, _ = ga_instance.best_solution(ga_instance.last_generation_fitness)
print(f"Mejor fitness: {solution_fitness}")
print(f"Días de inicio óptimos: {solution}")

# Graficar
ga_instance.plot_fitness()
plt.show()

In [ ]:
pip install deap numpy matplotlib

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt

# Datos del problema
TAREAS_OBRA = {
    0: (5, 4),   # Cimientos: 5 días, 4 operarios
    1: (7, 6),   # Muros: 7 días, 6 operarios
    2: (4, 5),   # Techo: 4 días, 5 operarios
    3: (6, 3),   # Instalaciones: 6 días, 3 operarios
    4: (5, 4),   # Revoque: 5 días, 4 operarios
    5: (4, 3),   # Pisos: 4 días, 3 operarios
    6: (3, 2),   # Pintura: 3 días, 2 operarios
    7: (2, 3),   # Aberturas: 2 días, 3 operarios
    8: (4, 2),   # Electricidad: 4 días, 2 operarios
    9: (2, 2)    # Final de obra: 2 días, 2 operarios
}

MAX_OPERARIOS = 10
NUM_TAREAS = len(TAREAS_OBRA)
MAX_DIAS = sum(duracion for duracion, _ in TAREAS_OBRA.values())

# Definir tipos de fitness e individuo
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimizar
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Atributo: día de inicio (0 a 30)
toolbox.register("dia_inicio", random.randint, 0, 30)

# Estructura del individuo (10 tareas)
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.dia_inicio, n=NUM_TAREAS)

# Población
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluar_cronograma(individual):
    """
    Calcula el makespan y verifica la restricción de operarios
    """
    operarios_por_dia = np.zeros(MAX_DIAS + 10)
    makespan = 0

    # Calcular uso de operarios por día
    for i in range(NUM_TAREAS):
        dia_inicio = int(individual[i])
        duracion, operarios_nec = TAREAS_OBRA[i]
        dia_fin = dia_inicio + duracion

        if dia_fin > makespan:
            makespan = dia_fin

        for d in range(dia_inicio, dia_fin):
            if d < len(operarios_por_dia):
                operarios_por_dia[d] += operarios_nec

    # Verificar restricción de operarios
    max_operarios_usados = np.max(operarios_por_dia)

    # Penalización drástica si excede el límite
    if max_operarios_usados > MAX_OPERARIOS:
        makespan = makespan * 1000  # Penalización grande

    return (makespan,)

toolbox.register("evaluate", evaluar_cronograma)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=0, up=30, indpb=0.1)
toolbox.register("select", tools.selTournament, tournsize=3)

def main():
    random.seed(42)

    # Crear población inicial
    pop = toolbox.population(n=100)

    # Estadísticas
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    # Ejecutar algoritmo genético
    print("Iniciando optimización con DEAP...")
    pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2,
                                    ngen=100, stats=stats, halloffame=hof,
                                    verbose=True)

    # Mejores resultados
    best_ind = hof[0]
    print(f"\n{'='*50}")
    print("MEJOR SOLUCIÓN ENCONTRADA")
    print(f"{'='*50}")
    print(f"Días de inicio: {best_ind}")
    print(f"Makespan: {best_ind.fitness.values[0]:.0f} días")

    # Verificar y mostrar cronograma detallado
    operarios_por_dia = np.zeros(MAX_DIAS + 10)
    makespan = 0

    print(f"\n{'Tarea':<10} {'Inicio':<10} {'Fin':<10} {'Duración':<10} {'Operarios':<10}")
    print("-" * 60)

    for i in range(NUM_TAREAS):
        dia_inicio = int(best_ind[i])
        duracion, operarios_nec = TAREAS_OBRA[i]
        dia_fin = dia_inicio + duracion

        if dia_fin > makespan:
            makespan = dia_fin

        for d in range(dia_inicio, dia_fin):
            operarios_por_dia[d] += operarios_nec

        print(f"{i:<10} {dia_inicio:<10} {dia_fin:<10} {duracion:<10} {operarios_nec:<10}")

    print("-" * 60)
    print(f"TIEMPO TOTAL (Makespan): {makespan} días")
    print(f"Máximo de operarios/día: {int(np.max(operarios_por_dia))} (Límite: {MAX_OPERARIOS})")

    # Graficar convergencia
    gen = log.select("gen")
    avg_fitness = log.select("avg")
    min_fitness = log.select("min")

    plt.figure(figsize=(10, 6))
    plt.plot(gen, avg_fitness, label='Fitness Promedio', linewidth=2)
    plt.plot(gen, min_fitness, label='Mejor Fitness', linewidth=2, color='red')
    plt.xlabel('Generación', fontsize=12)
    plt.ylabel('Makespan (días)', fontsize=12)
    plt.title('Convergencia del Algoritmo Genético', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Graficar uso de operarios por día
    dias_trabajo = range(makespan)
    operarios_usados = [operarios_por_dia[d] for d in dias_trabajo]

    plt.figure(figsize=(12, 5))
    plt.bar(dias_trabajo, operarios_usados, color='skyblue', edgecolor='blue')
    plt.axhline(y=MAX_OPERARIOS, color='red', linestyle='--', linewidth=2,
                label=f'Límite ({MAX_OPERARIOS} operarios)')
    plt.xlabel('Día', fontsize=12)
    plt.ylabel('Número de Operarios', fontsize=12)
    plt.title('Distribución de Operarios por Día', fontsize=14)
    plt.xticks(range(0, makespan, 2))
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

    return best_ind, pop

if __name__ == "__main__":
    main()